In [ ]:
import os
import json
import warnings
import torch
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

NVIDIA A100 80GB PCIe MIG 4g.40gb


In [3]:
for root, dirs, files in os.walk('/home/serfani/serfani_data1/E3OMA1950/'):
    
    sorted_files = sorted(files)
    list1 = [os.path.join(root, file) for file in sorted_files if file.split(".")[1] == 'aijlh1E33oma_ai']   # Velocity Fields (time, level, lat, lon)
    list2 = [os.path.join(root, file) for file in sorted_files if file.split(".")[1] == 'cijh1E33oma_ai']    # Precipitation (time, lat, lon)
    list3 = [os.path.join(root, file) for file in sorted_files if file.split(".")[1] == 'taijh1E33oma_ai']   # Sea Salt Src (time, lat, lon)
    list4 = [os.path.join(root, file) for file in sorted_files if file.split(".")[1] == 'tNDaijh1E33oma_ai'] # Clay-BCB Src (time, lat, lon)
    list5 = [os.path.join(root, file) for file in sorted_files if file.split(".")[1] == 'taijlh1E33oma_ai']  # Aerosols Mixing Ratio (time, level, lat, lon)


### Combine variables into a unique Zarr file

In [4]:
ds1 = xr.open_mfdataset(list1, combine='nested', concat_dim='time').isel(level=0, drop=True)
ds2 = xr.open_mfdataset(list2, combine='nested', concat_dim='time')[['prec']]
ds4 = xr.open_mfdataset(list4, combine='nested', concat_dim='time')[['BCB_biomass_src']]
ds5 = xr.open_mfdataset(list5, combine='nested', concat_dim='time').isel(level=0, drop=True)[['BCB']]

In [ ]:
warnings.filterwarnings("ignore", message="Converting a CFTimeIndex with dates from a non-standard calendar")
ds1 = ds1.indexes['time'].to_datetimeindex()  # 365 x 2 x 48 -> 35040
ds2 = ds2.indexes['time'].to_datetimeindex()  # 365 x 48 -> 17520
ds4 = ds4.indexes['time'].to_datetimeindex()  # 365 x 48 -> 17520
ds5 = ds5.indexes['time'].to_datetimeindex()

In [6]:
merged_ds = xr.merge([ds1, ds2, ds4, ds5])
merged_ds

<xarray.Dataset> Size: 13GB
Dimensions:          (time: 35040, lat: 90, lon: 144)
Coordinates:
  * lon              (lon) float32 576B -178.8 -176.2 -173.8 ... 176.2 178.8
  * lat              (lat) float32 360B -90.0 -87.0 -85.0 ... 85.0 87.0 90.0
  * time             (time) object 280kB 1950-01-01 00:15:00 ... 1951-12-31 2...
Data variables:
    axyp             (time, lat, lon) float32 2GB dask.array<chunksize=(48, 90, 144), meta=np.ndarray>
    u                (time, lat, lon) float32 2GB dask.array<chunksize=(48, 90, 144), meta=np.ndarray>
    v                (time, lat, lon) float32 2GB dask.array<chunksize=(48, 90, 144), meta=np.ndarray>
    omega            (time, lat, lon) float32 2GB dask.array<chunksize=(48, 90, 144), meta=np.ndarray>
    prec             (time, lat, lon) float32 2GB dask.array<chunksize=(48, 90, 144), meta=np.ndarray>
    BCB_biomass_src  (time, lat, lon) float32 2GB dask.array<chunksize=(48, 90, 144), meta=np.ndarray>
    BCB              (time, lat, lon) float32 2GB dask.array<chunksize=(48, 90, 144), meta=np.ndarray>
Attributes:
    xlabel:   E33oma_ai (OMA rundeck purely for initial coding of washout/aqc...

In [ ]:
zarr_path = '/home/serfani/serfani_data1/E3OMA1950.zarr'
merged_ds.to_zarr(zarr_path)

### Calculation of descriptive statistics of input/output variables

In [3]:
data1 = list()
ds1  = xr.open_mfdataset(list1[:365]).isel(level=0)
data1.append([ds1['u'].mean().values.item(), ds1['u'].std().values.item(), ds1['u'].min().values.item(), ds1['u'].max().values.item()])
data1.append([ds1['v'].mean().values.item(), ds1['v'].std().values.item(), ds1['v'].min().values.item(), ds1['v'].max().values.item()])
data1.append([ds1['omega'].mean().values.item(), ds1['omega'].std().values.item(), ds1['omega'].min().values.item(), ds1['omega'].max().values.item()])

ds2  = xr.open_mfdataset(list2[:365])
data1.append([ds2['prec'].mean().values.item(), ds2['prec'].std().values.item(), ds2['prec'].min().values.item(), ds2['prec'].max().values.item()])

ds3  = xr.open_mfdataset(list3[:365])
data1.append([ds3['seasalt1_ocean_src'].mean().values.item(), ds3['seasalt1_ocean_src'].std().values.item(), ds3['seasalt1_ocean_src'].min().values.item(), ds3['seasalt1_ocean_src'].max().values.item()])

ds4  = xr.open_mfdataset(list4[:365])
data1.append([ds4['Clay_emission'].mean().values.item(), ds4['Clay_emission'].std().values.item(), ds4['Clay_emission'].min().values.item(), ds4['Clay_emission'].max().values.item()])
data1.append([ds4['BCB_biomass_src'].mean().values.item(), ds4['BCB_biomass_src'].std().values.item(), ds4['BCB_biomass_src'].min().values.item(), ds4['BCB_biomass_src'].max().values.item()])

ds5  = xr.open_mfdataset(list5[:365]).isel(level=0)
data1.append([ds5['seasalt1'].mean().values.item(), ds5['seasalt1'].std().values.item(), ds5['seasalt1'].min().values.item(), ds5['seasalt1'].max().values.item()])
data1.append([ds5['Clay'].mean().values.item(), ds5['Clay'].std().values.item(), ds5['Clay'].min().values.item(), ds5['Clay'].max().values.item()])
data1.append([ds5['BCB'].mean().values.item(), ds5['BCB'].std().values.item(), ds5['BCB'].min().values.item(), ds5['BCB'].max().values.item()])


In [4]:
data2 = list()
ds1  = xr.open_mfdataset(list1[:365]).isel(level=0)
u = np.array(np.ma.log10(ds1['u'].values))
data2.append([u.mean(), u.std(), u.min(), u.max()])

v = np.array(np.ma.log10(ds1['v'].values))
data2.append([v.mean(), v.std(), v.min(), v.max()])

w = np.array(np.ma.log10(ds1['omega'].values))
data2.append([w.mean(), w.std(), w.min(), w.max()])


ds2  = xr.open_mfdataset(list2[:365])
p = np.array(np.ma.log10(ds2['prec'].values))
data2.append([p.mean(), p.std(), p.min(), p.max()])


ds3  = xr.open_mfdataset(list3[:365])
ss_src = np.array(np.ma.log10(ds3['seasalt1_ocean_src'].values))
data2.append([ss_src.mean(), ss_src.std(), ss_src.min(), ss_src.max()])


ds4  = xr.open_mfdataset(list4[:365])
c_src = np.array(np.ma.log10(ds4['Clay_emission'].values))
data2.append([c_src.mean(), c_src.std(), c_src.min(), c_src.max()])

bc_src = np.array(np.ma.log10(ds4['BCB_biomass_src'].values))
data2.append([bc_src.mean(), bc_src.std(), bc_src.min(), bc_src.max()])


ds5  = xr.open_mfdataset(list5[:365]).isel(level=0)
ss_conc = np.array(np.ma.log10(ds5['seasalt1'].values))
data2.append([ss_conc.mean(), ss_conc.std(), ss_conc.min(), ss_conc.max()])

c_conc= np.array(np.ma.log10(ds5['Clay'].values))
data2.append([c_conc.mean(), c_conc.std(), c_conc.min(), c_conc.max()])

bc_conc = np.array(np.ma.log10(ds5['BCB'].values))
data2.append([bc_conc.mean(), bc_conc.std(), bc_conc.min(), bc_conc.max()])

In [ ]:
import json

data = [data1, data2]

variables = ['u', 'v', 'w', 'prec', 'ss_src', 'c_src', 'bc_src', 'ss_conc', 'c_conc', 'bc_conc']

result = {'set1': {}, 'set2': {}}

for idx1, key in enumerate(result.keys()):
    for idx2, variable in enumerate(variables):
        mean, std, min_val, max_val = data[idx1][idx2]
        result[key][variable] = {
            'mean': float(mean),
            'std':  float(std),
            'min':  float(min_val),
            'max':  float(max_val)
        }

with open('statistics/1950.json', 'a') as jf:
    json.dump(result, jf, indent=4)
